# Cell2Cell Churn EDA
**Dataset**: Duke University / Teradata Center for CRM  
**Train**: 51,047 rows × 58 features | **Test**: 20,000 rows × 58 features  
**Target**: `Churn` — 71.2% No / 28.8% Yes (imbalanced)

### Objectives
1. Understand the target distribution and class imbalance
2. Identify missing value patterns
3. Explore numerical and categorical feature distributions
4. Find top churn drivers via correlation + univariate analysis
5. Validate SMOTE strategy and feature engineering decisions

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from configs.settings import settings
from src.churn.data.preprocess import load_raw, basic_clean, get_feature_columns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Imports OK')

## 1. Load Data

In [ ]:
train_raw, test_raw = load_raw()
print(f'Train : {train_raw.shape}')
print(f'Test  : {test_raw.shape}')
train_raw.head(3)

In [ ]:
train, caps = basic_clean(train_raw.copy())
test, _     = basic_clean(test_raw.copy(), caps=caps)
TARGET = settings.data.target_col
print(train.dtypes.value_counts())

## 2. Target Distribution

In [ ]:
churn_counts = train[TARGET].value_counts()
labels = ['No Churn (0)', 'Churn (1)']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar
axes[0].bar(labels, churn_counts.values, color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Churn Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=11)

# Pie
axes[1].pie(churn_counts.values, labels=labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Churn Split')

plt.suptitle('Cell2Cell — Target Variable (Train Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_target_dist.png', dpi=150)
plt.show()

print(f'\nImbalance ratio (majority/minority): {churn_counts[0]/churn_counts[1]:.2f}x')

## 3. Missing Values

In [ ]:
missing = train_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train_raw) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df)

fig, ax = plt.subplots(figsize=(10, 5))
missing_pct.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('Missing Values by Feature (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/eda_missing.png', dpi=150)
plt.show()

## 4. Numerical Feature Distributions by Churn

In [ ]:
num_cols, cat_cols = get_feature_columns(train)
print(f'Numeric: {len(num_cols)} | Categorical: {len(cat_cols)}')

# Top 12 numeric features
plot_cols = num_cols[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        data = train[train[TARGET] == label][col].dropna()
        axes[i].hist(data, bins=30, alpha=0.6, color=color,
                     label=f'{"No Churn" if label==0 else "Churn"}')
    axes[i].set_title(col, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle('Numeric Feature Distributions by Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_numeric_dist.png', dpi=150)
plt.show()

## 5. Churn Rate by Categorical Features

In [ ]:
# Churn rate per category value (top 6 categorical cols)
plot_cat = [c for c in cat_cols if c in train_raw.columns][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cat):
    churn_rate = train_raw.groupby(col)[TARGET].apply(
        lambda x: (x == 'Yes').mean() if x.dtype == object else x.mean()
    ).sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=axes[i], color='tomato', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Churn Rate by {col}', fontsize=10)
    axes[i].set_ylabel('Churn Rate')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=30, ha='right')
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.suptitle('Churn Rate by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_categorical_churn.png', dpi=150)
plt.show()

## 6. Correlation Heatmap (Numeric Features)

In [ ]:
corr_cols = num_cols[:20]  # top 20 numeric
corr = train[corr_cols + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdBu_r', vmin=-1, vmax=1,
    linewidths=0.4, annot=False, fmt='.2f',
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap (Numeric Features + Target)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_correlation.png', dpi=150)
plt.show()

# Top features correlated with Churn
churn_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print('\nTop 15 features correlated with Churn:')
print(churn_corr.head(15).to_string())

## 7. Churn Drivers — Interactive Plotly

In [ ]:
# Interactive box plots for top churn-correlated numeric features
top_features = churn_corr.head(8).index.tolist()
plot_df = train[top_features + [TARGET]].copy()
plot_df[TARGET] = plot_df[TARGET].map({0: 'No Churn', 1: 'Churn'})

fig = make_subplots(rows=2, cols=4, subplot_titles=top_features)

colors = {'No Churn': '#5b9bd5', 'Churn': '#e06c75'}
for i, feat in enumerate(top_features):
    row, col = divmod(i, 4)
    for label in ['No Churn', 'Churn']:
        data = plot_df[plot_df[TARGET] == label][feat].dropna()
        fig.add_trace(
            go.Box(y=data, name=label, marker_color=colors[label],
                   showlegend=(i == 0), legendgroup=label),
            row=row+1, col=col+1
        )

fig.update_layout(
    title_text='Top Churn-Correlated Features — Distribution by Class',
    height=600, boxmode='group',
    template='plotly_white',
)
fig.show()

## 8. Monthly Tenure vs Churn

In [ ]:
if 'MonthsInService' in train.columns:
    fig = px.histogram(
        train.assign(Churn=train[TARGET].map({0:'No Churn',1:'Churn'})),
        x='MonthsInService', color='Churn',
        barmode='overlay', opacity=0.7,
        color_discrete_map={'No Churn':'steelblue','Churn':'tomato'},
        title='Months in Service Distribution by Churn',
        nbins=40
    )
    fig.update_layout(template='plotly_white')
    fig.show()
    
    # Churn rate by tenure bucket
    train['tenure_bucket'] = pd.cut(train['MonthsInService'],
                                     bins=[0,3,6,12,24,36,60,200],
                                     labels=['0-3','4-6','7-12','13-24','25-36','37-60','60+'])
    churn_by_tenure = train.groupby('tenure_bucket')[TARGET].mean()
    print('Churn rate by tenure (months):')
    print(churn_by_tenure.to_string())
    train.drop(columns=['tenure_bucket'], inplace=True)
else:
    print('MonthsInService not in dataset')

## 9. Key EDA Takeaways

| Finding | Implication |
|---------|-------------|
| 28.8% churn rate — imbalanced | Use SMOTE + `scale_pos_weight` in XGBoost |
| 15 features with missing values | Median/mode imputation in pipeline |
| `HandsetPrice` has most nulls | Treat missing as its own category |
| Short-tenure customers churn more | `IsNewCustomer` interaction feature |
| High `DroppedCalls` correlates with churn | `DropRate` ratio feature |
| `MadeCallToRetentionTeam` is a strong signal | Binary flag — keep as-is |
| `CreditRating` shows churn gradient | Ordinal encode |
| Decision threshold = 0.40 (not 0.50) | Recall > Precision for business cost |
